# Spaceship Titanic - Kaggle Competition

**Goal**: Predict which passengers were transported to an alternate dimension

This notebook includes:
- Exploratory Data Analysis
- Feature Engineering
- Model Training: XGBoost, LightGBM, Logistic Regression
- Hyperparameter Optimization with Optuna + CV
- Submission File Generation

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)  

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")

c:\Users\matth\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


## 2. MLflow Setup

In [ ]:
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"   
EXPERIMENT_NAME     = "spaceship_titanic"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

# Cross-validation strategy
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"MLflow Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment          : {EXPERIMENT_NAME}")
print(f"CV Strategy         : StratifiedKFold(n_splits={N_SPLITS}, shuffle=True)")

2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/22 22:44:45 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/22 22:44:46 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/22 22:44:46 INFO alembic.runtime.migration: Will assume non-transactional DDL.


MLflow Tracking URI : sqlite:///mlflow.db
Experiment          : spaceship_titanic
CV Strategy         : StratifiedKFold(n_splits=5, shuffle=True)


## 3. Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
print(f"Training data shape : {train_df.shape}")
train_df.head()

Training data shape : (8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## 4. Feature Engineering

In [ ]:
def feature_engineering(df):
    """Apply comprehensive feature engineering."""
    df = df.copy()

    # Cabin features
    df['Deck']      = df['Cabin'].apply(lambda x: x.split('/')[0] if pd.notna(x) else 'Unknown')
    df['Cabin_num'] = df['Cabin'].apply(lambda x: x.split('/')[1] if pd.notna(x) else -1).astype(float)
    df['Side']      = df['Cabin'].apply(lambda x: x.split('/')[2] if pd.notna(x) else 'Unknown')

    # Group features
    df['Group']      = df['PassengerId'].apply(lambda x: x.split('_')[0])
    df['Group_size'] = df.groupby('Group')['Group'].transform('count')
    df['Solo']       = (df['Group_size'] == 1).astype(int)

    # Family features
    df['LastName']    = df['Name'].apply(lambda x: x.split()[-1] if pd.notna(x) else 'Unknown')
    df['Family_size'] = df.groupby('LastName')['LastName'].transform('count')

    # Spending features
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpending'] = df[spending_cols].sum(axis=1)
    df['HasSpending']   = (df['TotalSpending'] > 0).astype(int)
    df['NoSpending']    = (df['TotalSpending'] == 0).astype(int)
    for col in spending_cols:
        df[f'{col}_ratio'] = df[col] / (df['TotalSpending'] + 1)

    # Age features
    df['Age_group']       = pd.cut(df['Age'], bins=[0, 12, 18, 30, 50, 100],
                                   labels=['Child', 'Teen', 'Young_Adult', 'Adult', 'Senior'])
    df['Age_group']       = df['Age_group'].astype(str)
    df['Age_missing']     = df['Age'].isna().astype(int)
    df['CryoSleep_missing'] = df['CryoSleep'].isna().astype(int)

    return df


def preprocess_data(df, is_train=True):
    """Encode & select features for modelling."""
    df = df.copy()

    categorical_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'Age_group']
    numerical_features   = (
        ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
         'Cabin_num', 'Group_size', 'Solo', 'Family_size', 'TotalSpending',
         'HasSpending', 'NoSpending', 'Age_missing', 'CryoSleep_missing'] +
        [col for col in df.columns if '_ratio' in col]
    )

    for col in categorical_features:
        df[col] = df[col].fillna('Unknown')
    for col in numerical_features:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    for col in categorical_features:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    feature_columns = categorical_features + numerical_features
    X = df[feature_columns]

    if is_train:
        y = df['Transported'].astype(int)
        return X, y, feature_columns
    return X, feature_columns


train_df = feature_engineering(train_df)
X, y, feature_columns = preprocess_data(train_df, is_train=True)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Total features      : {len(feature_columns)}")
print(f"Train set           : {X_train.shape}")
print(f"Validation set      : {X_val.shape}")
print(f"Feature list        : {feature_columns}")

Total features      : 27
Train set           : (6954, 27)
Validation set      : (1739, 27)
Feature list        : ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'Age_group', 'Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Cabin_num', 'Group_size', 'Solo', 'Family_size', 'TotalSpending', 'HasSpending', 'NoSpending', 'Age_missing', 'CryoSleep_missing', 'RoomService_ratio', 'FoodCourt_ratio', 'ShoppingMall_ratio', 'Spa_ratio', 'VRDeck_ratio']


## 5. Helper: MLflow Logging Function



In [ ]:
def log_cv_run(
    run_name: str,
    stage: str,           
    model_type: str,    
    params: dict,
    cv_scores: np.ndarray,
    val_acc: float,
    val_auc: float,
    extra_tags: dict = None,
):
    """
    Log satu run MLflow lengkap:
      - params          : semua hyperparameter
      - cv_fold_accuracy: akurasi tiap fold (step = fold index)
      - cv_accuracy_mean / cv_accuracy_std
      - val_accuracy / val_roc_auc
    Returns run_id.
    """
    with mlflow.start_run(run_name=run_name) as run:
        # Tags
        mlflow.set_tag("stage",      stage)
        mlflow.set_tag("model_type", model_type)
        mlflow.set_tag("cv_folds",   str(len(cv_scores)))
        if extra_tags:
            for k, v in extra_tags.items():
                mlflow.set_tag(k, str(v))

        # Parameters 
        mlflow.log_params(params)

        # CV metrics per fold 
        for fold_i, score in enumerate(cv_scores):
            mlflow.log_metric("cv_fold_accuracy", score, step=fold_i)

        # Aggregate CV metrics 
        mlflow.log_metric("cv_accuracy_mean", float(cv_scores.mean()))
        mlflow.log_metric("cv_accuracy_std",  float(cv_scores.std()))
        mlflow.log_metric("cv_accuracy_min",  float(cv_scores.min()))
        mlflow.log_metric("cv_accuracy_max",  float(cv_scores.max()))

        #  Validation metrics 
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_roc_auc",  val_auc)

        run_id = run.info.run_id

    return run_id


def make_optuna_callback(parent_run_id: str, model_name: str):
    """
    Buat Optuna callback yang menyimpan setiap trial
    sebagai MLflow child run di bawah parent_run_id.
    """
    def callback(study, trial):
        with mlflow.start_run(
            run_name=f"{model_name}_trial_{trial.number:03d}",
            nested=True,
            tags={"mlflow.parentRunId": parent_run_id}
        ):
            mlflow.set_tag("stage",        "optuna_trial")
            mlflow.set_tag("model_type",   model_name)
            mlflow.set_tag("trial_number", str(trial.number))
            mlflow.set_tag("trial_state",  str(trial.state))
            mlflow.log_params(trial.params)
            mlflow.log_metric("cv_accuracy", trial.value)
        
            mlflow.log_metric("is_best", 1 if trial.value == study.best_value else 0)
    return callback



---
## 6. Baseline Models (3 models)


In [ ]:
baseline_configs = [
    {
        "name"      : "XGBoost_Baseline",
        "model_type": "XGBoost",
        "model"     : xgb.XGBClassifier(
                          random_state=RANDOM_STATE,
                          use_label_encoder=False,
                          eval_metric="logloss",
                          n_jobs=-1),
        "params"    : {"random_state": RANDOM_STATE,
                       "use_label_encoder": False,
                       "eval_metric": "logloss"},
    },
    {
        "name"      : "LightGBM_Baseline",
        "model_type": "LightGBM",
        "model"     : lgb.LGBMClassifier(
                          random_state=RANDOM_STATE,
                          verbose=-1,
                          n_jobs=-1),
        "params"    : {"random_state": RANDOM_STATE, "verbose": -1},
    },
    {
        "name"      : "LogisticRegression_Baseline",
        "model_type": "LogisticRegression",
        "model"     : LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        "params"    : {"random_state": RANDOM_STATE, "max_iter": 1000},
    },
]

BASELINE_RUN_NAMES = {
    "XGBoost"           : "Baseline_XGB",
    "LightGBM"          : "Baseline_LGBM",
    "LogisticRegression": "Baseline_LR",
}

baseline_results = {}

print("=" * 60)
print("BASELINE TRAINING")
print("=" * 60)

for cfg in baseline_configs:
    print(f"\n▶ {cfg['name']}")
    model = cfg["model"]

    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()

    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_acc  = accuracy_score(y_val, val_pred)
    val_auc  = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

    key = cfg["model_type"]
    baseline_results[key] = {"cv_mean": cv_mean, "cv_std": cv_std,
                              "val_acc": val_acc, "val_auc": val_auc}

    print(f"   CV per fold : {[f'{s:.4f}' for s in cv_scores]}")
    print(f"   CV Accuracy : {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"   Val Accuracy: {val_acc:.4f}  |  Val AUC: {val_auc:.4f}")

    run_name = BASELINE_RUN_NAMES[key]

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("stage",        "baseline")
        mlflow.set_tag("model_type",   key)
        mlflow.set_tag("cv_folds",     str(len(cv_scores)))
        mlflow.set_tag("optimization", "none")

        mlflow.log_params(cfg["params"])

        for fold_i, score in enumerate(cv_scores):
            mlflow.log_metric("cv_fold_accuracy", score, step=fold_i)
        mlflow.log_metric("cv_accuracy_mean", float(cv_mean))
        mlflow.log_metric("cv_accuracy_std",  float(cv_std))
        mlflow.log_metric("cv_accuracy_min",  float(cv_scores.min()))
        mlflow.log_metric("cv_accuracy_max",  float(cv_scores.max()))
        mlflow.log_metric("val_accuracy",     val_acc)
        mlflow.log_metric("val_roc_auc",      val_auc)

    
        if key == "XGBoost":
            mlflow.xgboost.log_model(model, artifact_path="model")
        elif key == "LightGBM":
            mlflow.lightgbm.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")

        baseline_results[key]["run_id"] = run.info.run_id
        print(f"   MLflow run_id : {run.info.run_id}")

print("\n" + "=" * 60)
print("BASELINE RESULTS SUMMARY")
print("=" * 60)
for mn, res in sorted(baseline_results.items(), key=lambda x: -x[1]["val_acc"]):
    print(f"  {mn:<25} Val Acc={res['val_acc']:.4f}  CV={res['cv_mean']:.4f}±{res['cv_std']:.4f}")


BASELINE TRAINING

▶ XGBoost_Baseline
   CV per fold : ['0.8212', '0.8108', '0.7993', '0.8021', '0.7929']
   CV Accuracy : 0.8052 ± 0.0098
   Val Accuracy: 0.7993  |  Val AUC: 0.9025


2026/02/22 22:44:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 22:45:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


   MLflow run_id : 22bee7177e6445beae532c353772b85b

▶ LightGBM_Baseline


2026/02/22 22:45:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 22:45:05 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.


   CV per fold : ['0.8177', '0.8051', '0.8074', '0.8188', '0.8055']
   CV Accuracy : 0.8109 ± 0.0061
   Val Accuracy: 0.8108  |  Val AUC: 0.9089


...

   MLflow run_id : 4c9c327c2a99463a99e9b9730808229c

▶ LogisticRegression_Baseline


2026/02/22 22:45:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   CV per fold : ['0.8010', '0.7746', '0.7936', '0.7917', '0.7796']
   CV Accuracy : 0.7881 ± 0.0096
   Val Accuracy: 0.7844  |  Val AUC: 0.8733


2026/02/22 22:45:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


   MLflow run_id : 997c935fda5d40be9cb5faffd666d6eb

BASELINE RESULTS SUMMARY
  LightGBM                  Val Acc=0.8108  CV=0.8109±0.0061
  XGBoost                   Val Acc=0.7993  CV=0.8052±0.0098
  LogisticRegression        Val Acc=0.7844  CV=0.7881±0.0096


---
## 7. Hyperparameter Optimization with Optuna

### 7.1 XGBoost 

In [ ]:
N_TRIALS_XGB = 50

def objective_xgb(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"    : trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
        "gamma"            : trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "use_label_encoder": False,
        "eval_metric"      : "logloss",
        "random_state"     : RANDOM_STATE,
        "n_jobs"           : -1,
    }
    model  = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return scores.mean()


print(f"Optimizing XGBoost ({N_TRIALS_XGB} trials)...")

study_xgb = optuna.create_study(
    direction  = "maximize",
    study_name = "xgboost_optuna",
    sampler    = TPESampler(seed=RANDOM_STATE),
)

with mlflow.start_run(run_name="XGBoost_Optuna") as xgb_parent_run:
    mlflow.set_tag("stage",        "optuna_parent")
    mlflow.set_tag("model_type",   "XGBoost")
    mlflow.set_tag("n_trials",     str(N_TRIALS_XGB))
    mlflow.set_tag("optimization", "optuna_tpe")

    xgb_callback = make_optuna_callback(xgb_parent_run.info.run_id, "XGBoost")
    study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB,
                       show_progress_bar=True, callbacks=[xgb_callback])

    best_params_xgb = study_xgb.best_params
    mlflow.log_params(best_params_xgb)
    mlflow.log_metric("best_cv_accuracy",   study_xgb.best_value)
    mlflow.log_metric("best_trial_number",  study_xgb.best_trial.number)
    mlflow.log_metric("n_trials_completed", len(study_xgb.trials))

    best_model_xgb = xgb.XGBClassifier(
        **best_params_xgb,
        use_label_encoder=False, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1)
    cv_best_xgb = cross_val_score(best_model_xgb, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    best_model_xgb.fit(X_train, y_train)
    val_acc_xgb = accuracy_score(y_val, best_model_xgb.predict(X_val))
    val_auc_xgb = roc_auc_score(y_val, best_model_xgb.predict_proba(X_val)[:, 1])

    for fold_i, score in enumerate(cv_best_xgb):
        mlflow.log_metric("cv_fold_accuracy", score, step=fold_i)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_xgb.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_xgb.std()))
    mlflow.log_metric("val_accuracy",     val_acc_xgb)
    mlflow.log_metric("val_roc_auc",      val_auc_xgb)

with mlflow.start_run(run_name="Final_XGB_Optimized") as xgb_final_run:
    mlflow.set_tag("stage",        "final")
    mlflow.set_tag("model_type",   "XGBoost")
    mlflow.set_tag("optimization", "optuna_tpe")

    mlflow.log_params(best_params_xgb)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_xgb.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_xgb.std()))
    mlflow.log_metric("val_accuracy",     val_acc_xgb)
    mlflow.log_metric("val_roc_auc",      val_auc_xgb)

    # Log model artifact
    mlflow.xgboost.log_model(best_model_xgb, artifact_path="model")
    xgb_final_run_id = xgb_final_run.info.run_id

print(f"Best XGBoost CV    : {study_xgb.best_value:.4f}")
print(f"Best trial #       : {study_xgb.best_trial.number}")
print(f"Best params        : {best_params_xgb}")
print(f"Val Accuracy       : {val_acc_xgb:.4f}  |  Val AUC: {val_auc_xgb:.4f}")
print(f"Final run_id       : {xgb_final_run_id}")


Optimizing XGBoost (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

2026/02/22 22:46:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 22:46:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best XGBoost CV    : 0.8112
Best trial #       : 46
Best params        : {'n_estimators': 959, 'max_depth': 3, 'learning_rate': 0.1281686657579565, 'subsample': 0.8504900029910624, 'colsample_bytree': 0.7750917610209548, 'min_child_weight': 6, 'gamma': 3.198286898753319, 'reg_alpha': 0.05365982014462911, 'reg_lambda': 0.00010112180975310941}
Val Accuracy       : 0.8097  |  Val AUC: 0.9044
Final run_id       : e55ff97d234048838e2d58c31601a493


### 7.2 LightGBM 

In [ ]:
N_TRIALS_LGB = 50

def objective_lgb(trial):
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 100, 1000),
        "max_depth"         : trial.suggest_int("max_depth", 3, 12),
        "learning_rate"     : trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "num_leaves"        : trial.suggest_int("num_leaves", 20, 300),
        "min_child_samples" : trial.suggest_int("min_child_samples", 5, 100),
        "subsample"         : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state"      : RANDOM_STATE,
        "verbose"           : -1,
        "n_jobs"            : -1,
    }
    model  = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return scores.mean()


print(f"Optimizing LightGBM ({N_TRIALS_LGB} trials)...")

study_lgb = optuna.create_study(
    direction  = "maximize",
    study_name = "lightgbm_optuna",
    sampler    = TPESampler(seed=RANDOM_STATE),
)

with mlflow.start_run(run_name="LightGBM_Optuna") as lgb_parent_run:
    mlflow.set_tag("stage",        "optuna_parent")
    mlflow.set_tag("model_type",   "LightGBM")
    mlflow.set_tag("n_trials",     str(N_TRIALS_LGB))
    mlflow.set_tag("optimization", "optuna_tpe")

    lgb_callback = make_optuna_callback(lgb_parent_run.info.run_id, "LightGBM")
    study_lgb.optimize(objective_lgb, n_trials=N_TRIALS_LGB,
                       show_progress_bar=True, callbacks=[lgb_callback])

    best_params_lgb = study_lgb.best_params
    mlflow.log_params(best_params_lgb)
    mlflow.log_metric("best_cv_accuracy",   study_lgb.best_value)
    mlflow.log_metric("best_trial_number",  study_lgb.best_trial.number)
    mlflow.log_metric("n_trials_completed", len(study_lgb.trials))

    best_model_lgb = lgb.LGBMClassifier(
        **best_params_lgb, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
    cv_best_lgb = cross_val_score(best_model_lgb, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    best_model_lgb.fit(X_train, y_train)
    val_acc_lgb = accuracy_score(y_val, best_model_lgb.predict(X_val))
    val_auc_lgb = roc_auc_score(y_val, best_model_lgb.predict_proba(X_val)[:, 1])

    for fold_i, score in enumerate(cv_best_lgb):
        mlflow.log_metric("cv_fold_accuracy", score, step=fold_i)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_lgb.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_lgb.std()))
    mlflow.log_metric("val_accuracy",     val_acc_lgb)
    mlflow.log_metric("val_roc_auc",      val_auc_lgb)

with mlflow.start_run(run_name="Final_LGBM_Optimized") as lgb_final_run:
    mlflow.set_tag("stage",        "final")
    mlflow.set_tag("model_type",   "LightGBM")
    mlflow.set_tag("optimization", "optuna_tpe")

    mlflow.log_params(best_params_lgb)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_lgb.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_lgb.std()))
    mlflow.log_metric("val_accuracy",     val_acc_lgb)
    mlflow.log_metric("val_roc_auc",      val_auc_lgb)

    mlflow.lightgbm.log_model(best_model_lgb, artifact_path="model")
    lgb_final_run_id = lgb_final_run.info.run_id

print(f"Best LightGBM CV   : {study_lgb.best_value:.4f}")
print(f"Best trial #       : {study_lgb.best_trial.number}")
print(f"Best params        : {best_params_lgb}")
print(f"Val Accuracy       : {val_acc_lgb:.4f}  |  Val AUC: {val_auc_lgb:.4f}")
print(f"Final run_id       : {lgb_final_run_id}")


Optimizing LightGBM (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

2026/02/22 22:49:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 22:49:53 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
2026/02/22 22:49:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best LightGBM CV   : 0.8148
Best trial #       : 28
Best params        : {'n_estimators': 210, 'max_depth': 10, 'learning_rate': 0.0470421419336879, 'num_leaves': 23, 'min_child_samples': 44, 'subsample': 0.7166813175357148, 'colsample_bytree': 0.9105942168984561, 'reg_alpha': 0.0012231535267654102, 'reg_lambda': 0.0022449942710649246}
Val Accuracy       : 0.8102  |  Val AUC: 0.9090
Final run_id       : 8e909aa5545f4544ab773430c8bfde70


### 7.3 Logistic Regression 

In [ ]:
N_TRIALS_LR = 30

def objective_lr(trial):
    params = {
        "C"           : trial.suggest_float("C", 0.001, 100, log=True),
        "penalty"     : trial.suggest_categorical("penalty", ["l1", "l2"]),
        "solver"      : trial.suggest_categorical("solver", ["liblinear", "saga"]),
        "max_iter"    : trial.suggest_int("max_iter", 100, 2000),
        "random_state": RANDOM_STATE,
    }
    model  = LogisticRegression(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return scores.mean()


print(f"Optimizing Logistic Regression ({N_TRIALS_LR} trials)...")

study_lr = optuna.create_study(
    direction  = "maximize",
    study_name = "logreg_optuna",
    sampler    = TPESampler(seed=RANDOM_STATE),
)

with mlflow.start_run(run_name="LogReg_Optuna") as lr_parent_run:
    mlflow.set_tag("stage",        "optuna_parent")
    mlflow.set_tag("model_type",   "LogisticRegression")
    mlflow.set_tag("n_trials",     str(N_TRIALS_LR))
    mlflow.set_tag("optimization", "optuna_tpe")

    lr_callback = make_optuna_callback(lr_parent_run.info.run_id, "LogisticRegression")
    study_lr.optimize(objective_lr, n_trials=N_TRIALS_LR,
                      show_progress_bar=True, callbacks=[lr_callback])

    best_params_lr = study_lr.best_params
    mlflow.log_params(best_params_lr)
    mlflow.log_metric("best_cv_accuracy",   study_lr.best_value)
    mlflow.log_metric("best_trial_number",  study_lr.best_trial.number)
    mlflow.log_metric("n_trials_completed", len(study_lr.trials))

    best_model_lr = LogisticRegression(**best_params_lr, random_state=RANDOM_STATE)
    cv_best_lr    = cross_val_score(best_model_lr, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    best_model_lr.fit(X_train, y_train)
    val_acc_lr = accuracy_score(y_val, best_model_lr.predict(X_val))
    val_auc_lr = roc_auc_score(y_val, best_model_lr.predict_proba(X_val)[:, 1])

    for fold_i, score in enumerate(cv_best_lr):
        mlflow.log_metric("cv_fold_accuracy", score, step=fold_i)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_lr.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_lr.std()))
    mlflow.log_metric("val_accuracy",     val_acc_lr)
    mlflow.log_metric("val_roc_auc",      val_auc_lr)

with mlflow.start_run(run_name="Final_LR_Optimized") as lr_final_run:
    mlflow.set_tag("stage",        "final")
    mlflow.set_tag("model_type",   "LogisticRegression")
    mlflow.set_tag("optimization", "optuna_tpe")

    mlflow.log_params(best_params_lr)
    mlflow.log_metric("cv_accuracy_mean", float(cv_best_lr.mean()))
    mlflow.log_metric("cv_accuracy_std",  float(cv_best_lr.std()))
    mlflow.log_metric("val_accuracy",     val_acc_lr)
    mlflow.log_metric("val_roc_auc",      val_auc_lr)

    mlflow.sklearn.log_model(best_model_lr, artifact_path="model")
    lr_final_run_id = lr_final_run.info.run_id

print(f"Best LR CV         : {study_lr.best_value:.4f}")
print(f"Best trial #       : {study_lr.best_trial.number}")
print(f"Best params        : {best_params_lr}")
print(f"Val Accuracy       : {val_acc_lr:.4f}  |  Val AUC: {val_auc_lr:.4f}")
print(f"Final run_id       : {lr_final_run_id}")


Optimizing Logistic Regression (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

2026/02/22 22:50:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 22:50:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Best LR CV         : 0.7931
Best trial #       : 18
Best params        : {'C': 0.006579496388600943, 'penalty': 'l1', 'solver': 'liblinear', 'max_iter': 485}
Val Accuracy       : 0.7959  |  Val AUC: 0.8686
Final run_id       : 81fc29caac2f4d33830635624e3b4979


---
## 8. Train Final Models with Best Parameters




In [ ]:
summary_data = [
    # Baseline
    {"Model": "XGBoost",            "Stage": "Baseline",
     "CV Mean": baseline_results["XGBoost"]["cv_mean"],
     "CV Std" : baseline_results["XGBoost"]["cv_std"],
     "Val Acc": baseline_results["XGBoost"]["val_acc"],
     "Val AUC": baseline_results["XGBoost"]["val_auc"],
     "run_id" : baseline_results["XGBoost"]["run_id"]},
    {"Model": "LightGBM",           "Stage": "Baseline",
     "CV Mean": baseline_results["LightGBM"]["cv_mean"],
     "CV Std" : baseline_results["LightGBM"]["cv_std"],
     "Val Acc": baseline_results["LightGBM"]["val_acc"],
     "Val AUC": baseline_results["LightGBM"]["val_auc"],
     "run_id" : baseline_results["LightGBM"]["run_id"]},
    {"Model": "LogisticRegression", "Stage": "Baseline",
     "CV Mean": baseline_results["LogisticRegression"]["cv_mean"],
     "CV Std" : baseline_results["LogisticRegression"]["cv_std"],
     "Val Acc": baseline_results["LogisticRegression"]["val_acc"],
     "Val AUC": baseline_results["LogisticRegression"]["val_auc"],
     "run_id" : baseline_results["LogisticRegression"]["run_id"]},
    # Final Optimized
    {"Model": "XGBoost",            "Stage": "Optuna",
     "CV Mean": cv_best_xgb.mean(), "CV Std": cv_best_xgb.std(),
     "Val Acc": val_acc_xgb,        "Val AUC": val_auc_xgb,
     "run_id" : xgb_final_run_id},
    {"Model": "LightGBM",           "Stage": "Optuna",
     "CV Mean": cv_best_lgb.mean(), "CV Std": cv_best_lgb.std(),
     "Val Acc": val_acc_lgb,        "Val AUC": val_auc_lgb,
     "run_id" : lgb_final_run_id},
    {"Model": "LogisticRegression", "Stage": "Optuna",
     "CV Mean": cv_best_lr.mean(),  "CV Std": cv_best_lr.std(),
     "Val Acc": val_acc_lr,         "Val AUC": val_auc_lr,
     "run_id" : lr_final_run_id},
]

summary_df = pd.DataFrame(summary_data).sort_values("Val Acc", ascending=False).reset_index(drop=True)

print("=" * 75)
print("FULL EXPERIMENT COMPARISON")
print("=" * 75)
print(summary_df.drop(columns="run_id").to_string(index=False, float_format="{:.4f}".format))

best_row        = summary_df.iloc[0]
best_model_name = f"{best_row['Model']} ({best_row['Stage']})"
print(f"\n🏆  Best: {best_model_name}  Val Acc={best_row['Val Acc']:.4f}  Val AUC={best_row['Val AUC']:.4f}")

with mlflow.start_run(run_name="_Summary_All_Models") as summary_run:
    mlflow.set_tag("stage",      "summary")
    mlflow.set_tag("best_model", best_model_name)
    for _, row in summary_df.iterrows():
        prefix = f"{row['Model']}_{row['Stage']}"
        mlflow.log_metric(f"{prefix}_cv_mean", row["CV Mean"])
        mlflow.log_metric(f"{prefix}_val_acc", row["Val Acc"])
        mlflow.log_metric(f"{prefix}_val_auc", row["Val AUC"])
    mlflow.log_metric("best_val_accuracy", float(best_row["Val Acc"]))
    mlflow.log_metric("best_val_auc",      float(best_row["Val AUC"]))

print(f"Summary run_id: {summary_run.info.run_id}")


FULL EXPERIMENT COMPARISON
             Model    Stage  CV Mean  CV Std  Val Acc  Val AUC
          LightGBM Baseline   0.8109  0.0061   0.8108   0.9089
          LightGBM   Optuna   0.8148  0.0029   0.8102   0.9090
           XGBoost   Optuna   0.8112  0.0070   0.8097   0.9044
           XGBoost Baseline   0.8052  0.0098   0.7993   0.9025
LogisticRegression   Optuna   0.7931  0.0054   0.7959   0.8686
LogisticRegression Baseline   0.7881  0.0096   0.7844   0.8733

🏆  Best: LightGBM (Baseline)  Val Acc=0.8108  Val AUC=0.9089
Summary run_id: 0d74f4666c98496497e02744d5ea680c


## 9 Model Management — Register & Champion Alias



In [ ]:

final_candidates = [
    {
        "model_type": "XGBoost",
        "run_id"    : xgb_final_run_id,
        "val_acc"   : val_acc_xgb,
        "val_auc"   : val_auc_xgb,
        "run_name"  : "Final_XGB_Optimized",
    },
    {
        "model_type": "LightGBM",
        "run_id"    : lgb_final_run_id,
        "val_acc"   : val_acc_lgb,
        "val_auc"   : val_auc_lgb,
        "run_name"  : "Final_LGBM_Optimized",
    },
    {
        "model_type": "LogisticRegression",
        "run_id"    : lr_final_run_id,
        "val_acc"   : val_acc_lr,
        "val_auc"   : val_auc_lr,
        "run_name"  : "Final_LR_Optimized",
    },
]

final_candidates.sort(key=lambda x: x["val_acc"], reverse=True)
best_candidate = final_candidates[0]

print("MODEL REGISTRY")
print("\nRanking 3 Final Models:")
for i, c in enumerate(final_candidates):
    marker = "🏆" if i == 0 else f"  {i+1}."
    print(f"  {marker}  {c['model_type']:<25} Val Acc={c['val_acc']:.4f}  Val AUC={c['val_auc']:.4f}")

print(f"\n→ Model terbaik: {best_candidate['model_type']} (Val Acc={best_candidate['val_acc']:.4f})")

MODEL_REGISTRY_NAME = "SpaceshipTitanicModel"
model_uri           = f"runs:/{best_candidate['run_id']}/model"

print(f"\nRegistering ke Model Registry...")
print(f"  Registry name : {MODEL_REGISTRY_NAME}")
print(f"  Source run_id : {best_candidate['run_id']}")
print(f"  Model URI     : {model_uri}")


registered_mv = mlflow.register_model(
    model_uri   = model_uri,
    name        = MODEL_REGISTRY_NAME,
)

print(f"\n✓ Registered: {MODEL_REGISTRY_NAME} v{registered_mv.version}")

client = mlflow.MlflowClient()

client.set_registered_model_alias(
    name    = MODEL_REGISTRY_NAME,
    alias   = "champion",
    version = registered_mv.version,
)

client.update_registered_model(
    name        = MODEL_REGISTRY_NAME,
    description = (
        f"Spaceship Titanic classifier. "
        f"Champion: {best_candidate['model_type']} "
        f"(Val Acc={best_candidate['val_acc']:.4f}, "
        f"Val AUC={best_candidate['val_auc']:.4f})"
    ),
)

client.update_model_version(
    name        = MODEL_REGISTRY_NAME,
    version     = registered_mv.version,
    description = (
        f"Optimized {best_candidate['model_type']} via Optuna. "
        f"CV Accuracy: {cv_best_lgb.mean() if best_candidate['model_type'] == 'LightGBM' else cv_best_xgb.mean() if best_candidate['model_type'] == 'XGBoost' else cv_best_lr.mean():.4f}. "
        f"Val Accuracy: {best_candidate['val_acc']:.4f}. "
        f"Val AUC: {best_candidate['val_auc']:.4f}."
    ),
)

print(f" Alias @champion → {MODEL_REGISTRY_NAME} v{registered_mv.version}")
print(f"MODEL REGISTRY SUMMARY")

print(f"  Registered model  : {MODEL_REGISTRY_NAME}")
print(f"  Version           : {registered_mv.version}")
print(f"  Alias             : @champion")
print(f"  Model type        : {best_candidate['model_type']}")
print(f"  Source run        : {best_candidate['run_name']}")
print(f"  Val Accuracy      : {best_candidate['val_acc']:.4f}")
print(f"  Val AUC           : {best_candidate['val_auc']:.4f}")


Registered model 'SpaceshipTitanicModel' already exists. Creating a new version of this model...
2026/02/22 22:50:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 8e909aa5545f4544ab773430c8bfde70 has no artifacts at artifact path 'model', registering model based on models:/m-71fb6e1067a4459eb61cde50b17ea41c instead


MODEL REGISTRY

Ranking 3 Final Models:
  🏆  LightGBM                  Val Acc=0.8102  Val AUC=0.9090
    2.  XGBoost                   Val Acc=0.8097  Val AUC=0.9044
    3.  LogisticRegression        Val Acc=0.7959  Val AUC=0.8686

→ Model terbaik: LightGBM (Val Acc=0.8102)

Registering ke Model Registry...
  Registry name : SpaceshipTitanicModel
  Source run_id : 8e909aa5545f4544ab773430c8bfde70
  Model URI     : runs:/8e909aa5545f4544ab773430c8bfde70/model

✓ Registered: SpaceshipTitanicModel v4
 Alias @champion → SpaceshipTitanicModel v4
MODEL REGISTRY SUMMARY
  Registered model  : SpaceshipTitanicModel
  Version           : 4
  Alias             : @champion
  Model type        : LightGBM
  Source run        : Final_LGBM_Optimized
  Val Accuracy      : 0.8102
  Val AUC           : 0.9090


Created version '4' of model 'SpaceshipTitanicModel'.
